In [0]:
# CELL 1: CONFIGURATION & SECURITY
# --------------------------------
# Define storage account details
storage_account_name = "dlpeopleanalytics2026"
container_name = "medallion-data"

# Authenticate using Azure Key Vault (Zero Trust approach)
storage_account_key = dbutils.secrets.get(scope="kv-secrets", key="storage-account-key")

# Configure Spark session to access Azure Blob Storage
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

In [0]:
# CELL 2: DATA EXTRACTION (RAW ZONE) - INCREMENTAL SETUP
# ------------------------------------------------------
# 1. Define the widget to receive the parameter from Azure Data Factory
dbutils.widgets.text("TargetDate", "20260731")

# 2. Capture the value
target_date = dbutils.widgets.get("TargetDate")
print(f"Executing Bronze Layer ingestion for target date: {target_date}")

# 3. Define base path for the raw layer
raw_base_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/raw"

# 4. Define specific file paths targeting ONLY the current execution date
raw_roles_path = f"{raw_base_path}/dim_role_{target_date}.csv"
raw_headcount_path = f"{raw_base_path}/headcount_{target_date}.csv"
raw_salaries_path = f"{raw_base_path}/reference_salaries_{target_date}.csv"
raw_survey_path = f"{raw_base_path}/climate_survey_{target_date}.csv"

# 5. Read standard CSV files into Spark DataFrames
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, IntegerType, DateType

# Matches exactly what's already in bronze_headcount_path (checked with printSchema())
headcount_schema = StructType([
    StructField("employee_id", StringType(), True),
    StructField("role_id", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("current_salary", DoubleType(), True),
    StructField("hire_date", DateType(), True),
    StructField("termination_date", DateType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("performance_rating", IntegerType(), True),
    StructField("snapshot_date", DateType(), True),
])

df_roles_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(raw_roles_path)
df_headcount_raw = spark.read.format("csv").option("header", "true").schema(headcount_schema).load(raw_headcount_path)
df_salaries_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(raw_salaries_path)

# 6. Read survey gracefully (since it only runs semi-annually, it might not exist this month)
try:
    df_survey_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(raw_survey_path)
    survey_exists = True
except Exception as e:
    print(f"Note: No climate survey file found for {target_date}. Skipping survey ingestion.")
    survey_exists = False

In [0]:
# CELL 3: DATA LOADING (BRONZE LAYER) - INCREMENTAL APPEND
# --------------------------------------------------------
from pyspark.sql.functions import current_timestamp, input_file_name, regexp_extract, to_date

# Define base path for the bronze layer
bronze_base_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/bronze"

# Define specific file paths for Delta tables
bronze_roles_path = f"{bronze_base_path}/dim_role"
bronze_headcount_path = f"{bronze_base_path}/headcount"
bronze_salaries_path = f"{bronze_base_path}/reference_salaries"
bronze_survey_path = f"{bronze_base_path}/climate_survey"

# Add metadata columns (snapshot_date is natively included in these CSVs)
df_roles_bronze = df_roles_raw.withColumn("ingestion_timestamp", current_timestamp())
df_headcount_bronze = df_headcount_raw.withColumn("ingestion_timestamp", current_timestamp())
df_salaries_bronze = df_salaries_raw.withColumn("ingestion_timestamp", current_timestamp())

# Write standard DataFrames to Bronze layer (APPEND)
df_roles_bronze.write.format("delta").mode("append").save(bronze_roles_path)
df_headcount_bronze.write.format("delta").mode("append").save(bronze_headcount_path)
df_salaries_bronze.write.format("delta").mode("append").save(bronze_salaries_path)

# Only process and append survey data if the file was found
if survey_exists:
    # Adding metadata and dynamically extracting snapshot_date from the filename just in case
    df_survey_bronze = df_survey_raw \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .withColumn("source_filename", input_file_name()) \
        .withColumn("date_string", regexp_extract("source_filename", r"(\d{8})", 1)) \
        .withColumn("snapshot_date", to_date("date_string", "yyyyMMdd")) \
        .drop("date_string")
        
    df_survey_bronze.write.format("delta").mode("append").save(bronze_survey_path)
    print("Survey data successfully appended.")

print("Bronze Layer incremental ingestion completed successfully.")

All raw files successfully appended to the Bronze layer with extended metadata.
